In [1]:
from pathlib import Path

In [2]:
IMAGE_DIR = Path(r"C:\Users\MHamzaRao\Desktop\Finding-Images-Through-Spoken-Language-THI-AVE-Team-Project\Image_Dataset\Flicker8k_Dataset")
AUDIO_DIR = Path(r"C:\Users\MHamzaRao\Desktop\Finding-Images-Through-Spoken-Language-THI-AVE-Team-Project\Audio_Dataset\flickr_audio")

CAPTION_MAP = Path(r"C:\Users\MHamzaRao\Desktop\Finding-Images-Through-Spoken-Language-THI-AVE-Team-Project\Audio_Dataset\wav2capt.txt")
SPEAKER_MAP = Path(r"C:\Users\MHamzaRao\Desktop\Finding-Images-Through-Spoken-Language-THI-AVE-Team-Project\Audio_Dataset\wav2spk.txt")

# Reports will be saved alongside your notebook
OUTPUT_DIR = Path.cwd() / "audit_report"
OUTPUT_DIR.mkdir(exist_ok=True)

for name, path in {
    "Images": IMAGE_DIR,
    "Audio": AUDIO_DIR,
    "Caption mapping": CAPTION_MAP,
    "Speaker mapping": SPEAKER_MAP,
}.items():
    print(f"{name}: {'FOUND' if path.exists() else 'NOT FOUND'}")
    print(f"  {path}")

Images: FOUND
  C:\Users\MHamzaRao\Desktop\Finding-Images-Through-Spoken-Language-THI-AVE-Team-Project\Image_Dataset\Flicker8k_Dataset
Audio: FOUND
  C:\Users\MHamzaRao\Desktop\Finding-Images-Through-Spoken-Language-THI-AVE-Team-Project\Audio_Dataset\flickr_audio
Caption mapping: FOUND
  C:\Users\MHamzaRao\Desktop\Finding-Images-Through-Spoken-Language-THI-AVE-Team-Project\Audio_Dataset\wav2capt.txt
Speaker mapping: FOUND
  C:\Users\MHamzaRao\Desktop\Finding-Images-Through-Spoken-Language-THI-AVE-Team-Project\Audio_Dataset\wav2spk.txt


In [3]:
image_files = [
    p for p in IMAGE_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png"}
]

audio_files = [
    p for p in AUDIO_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() == ".wav"
]

print("Images:", len(image_files))
print("Audio recordings:", len(audio_files))

Images: 8091
Audio recordings: 40000


In [4]:

import pandas as pd

# Build filename sets for fast lookup
image_names = {p.name for p in image_files}
audio_names = {p.name for p in audio_files}

rows = []

with CAPTION_MAP.open("r", encoding="utf-8-sig") as f:
    for line_no, line in enumerate(f, start=1):
        line = line.strip()

        if not line:
            continue

        parts = line.split()

        if len(parts) < 2:
            print(f"Skipping malformed line {line_no}: {line}")
            continue

        audio_name, image_name = parts[:2]

        # Skip header
        if audio_name.lower() in {"audio", "audio_file", "wav"}:
            continue

        rows.append({
            "audio_file": audio_name,
            "image_file": image_name,
            "caption_number": (
                parts[2].lstrip("#") if len(parts) >= 3 else None
            ),
        })

mapping_df = pd.DataFrame(rows)

mapping_df["audio_exists"] = mapping_df["audio_file"].isin(audio_names)
mapping_df["image_exists"] = mapping_df["image_file"].isin(image_names)

print("Mapping rows:", len(mapping_df))
print("Unique mapped audio files:", mapping_df["audio_file"].nunique())
print("Unique mapped images:", mapping_df["image_file"].nunique())
print("Rows with missing audio:", (~mapping_df["audio_exists"]).sum())
print("Rows with missing images:", (~mapping_df["image_exists"]).sum())

print("\nCaption-number distribution:")
print(mapping_df["caption_number"].value_counts(dropna=False).sort_index())

print("\nFirst 10 mapping rows:")
display(mapping_df.head(10))


Mapping rows: 40000
Unique mapped audio files: 40000
Unique mapped images: 8000
Rows with missing audio: 0
Rows with missing images: 0

Caption-number distribution:
caption_number
0    8000
1    8000
2    8000
3    8000
4    8000
Name: count, dtype: int64

First 10 mapping rows:


,audio_file,image_file,caption_number,audio_exists,image_exists
0,2571096893_694ce79768_1.wav,2571096893_694ce79768.jpg,1,True,True
1,2571096893_694ce79768_2.wav,2571096893_694ce79768.jpg,2,True,True
2,2570559405_dc93007f76_4.wav,2570559405_dc93007f76.jpg,4,True,True
3,2571096893_694ce79768_0.wav,2571096893_694ce79768.jpg,0,True,True
4,2572101672_4d699c8713_0.wav,2572101672_4d699c8713.jpg,0,True,True
5,2572101672_4d699c8713_1.wav,2572101672_4d699c8713.jpg,1,True,True
6,2571096893_694ce79768_3.wav,2571096893_694ce79768.jpg,3,True,True
7,2571096893_694ce79768_4.wav,2571096893_694ce79768.jpg,4,True,True
8,2572101672_4d699c8713_2.wav,2572101672_4d699c8713.jpg,2,True,True
9,2572101672_4d699c8713_3.wav,2572101672_4d699c8713.jpg,3,True,True


In [5]:
captions_per_image = (
    mapping_df.groupby("image_file")["audio_file"]
    .nunique()
)

print("Images represented in mapping:", len(captions_per_image))
print("\nNumber of audio captions per image:")
print(captions_per_image.value_counts().sort_index())

print("\nImages without exactly 5 captions:",
      (captions_per_image != 5).sum())

# 2. Check duplicates
duplicate_pairs = mapping_df.duplicated(
    subset=["audio_file", "image_file"]
)

print("\nDuplicate audio-image mapping rows:",
      duplicate_pairs.sum())

# 3. Extra images
mapped_images = set(mapping_df["image_file"])
all_images = {p.name for p in image_files}
extra_images = sorted(all_images - mapped_images)

print("\nImages not referenced by the audio mapping:",
      len(extra_images))
print("Examples:", extra_images[:10])


Images represented in mapping: 8000

Number of audio captions per image:
audio_file
5    8000
Name: count, dtype: int64

Images without exactly 5 captions: 0

Duplicate audio-image mapping rows: 0

Images not referenced by the audio mapping: 91
Examples: ['1155138244_859fd6e079.jpg', '1468103286_96a6e07029.jpg', '1479857177_9d4a6f38fd.jpg', '1643915227_9f48068772.jpg', '1797554350_20998753c0.jpg', '1808504612_3508f3c9bb.jpg', '199463720_329a802206.jpg', '2058091220_2087270068.jpg', '2087317114_cf06df5aa5.jpg', '2136455112_202c093ba4.jpg']


In [6]:
captions_per_image = mapping_df.groupby("image_file")["audio_file"].nunique()

print("Caption count distribution:")
print(captions_per_image.value_counts().sort_index())

print("\nImages without exactly 5 captions:",
      (captions_per_image != 5).sum())

print("\nDuplicate audio-image pairs:",
      mapping_df.duplicated(["audio_file", "image_file"]).sum())

print("Audio files mapped to multiple images:",
      (mapping_df.groupby("audio_file")["image_file"].nunique() > 1).sum())

Caption count distribution:
audio_file
5    8000
Name: count, dtype: int64

Images without exactly 5 captions: 0

Duplicate audio-image pairs: 0
Audio files mapped to multiple images: 0


In [7]:

from sklearn.model_selection import train_test_split
import pandas as pd

image_ids = sorted(mapping_df["image_file"].unique())

# 80% train, 20% temporary
train_ids, temp_ids = train_test_split(
    image_ids,
    test_size=0.20,
    random_state=42
)

val_ids, test_ids = train_test_split(
    temp_ids,
    test_size=0.50,
    random_state=42
)

split_lookup = {
    **{image: "train" for image in train_ids},
    **{image: "validation" for image in val_ids},
    **{image: "test" for image in test_ids},
}

dataset_df = mapping_df.copy()
dataset_df["split"] = dataset_df["image_file"].map(split_lookup)

# Save metadata only, not the underlying media
OUTPUT_DIR.mkdir(exist_ok=True)

dataset_df.to_csv(OUTPUT_DIR / "dataset_splits.csv", index=False)

print("Images per split:")
print(pd.Series(split_lookup).value_counts())

print("\nAudio-image pairs per split:")
print(dataset_df.groupby("split").size())

print("\nImages shared across splits (should be 0):")
image_split_counts = dataset_df.groupby("image_file")["split"].nunique()
print((image_split_counts > 1).sum())


Images per split:
train         6400
validation     800
test           800
Name: count, dtype: int64

Audio-image pairs per split:
split
test           4000
train         32000
validation     4000
dtype: int64

Images shared across splits (should be 0):
0
